# Capítulo 5 · Descenso por gradiente

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/03_descenso_por_gradiente.ipynb)

**Aprendizaje supervisado · Parte 3: cómo se encuentran los coeficientes**

### 📚 Índice
1. La idea
2. Un coeficiente, en 2D
3. Un paso a mano
4. Pseudocódigo
5. Una característica
6. La tasa de aprendizaje
7. Dos características, en 3D
8. Chuleta resumen


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep", 4)

SEED = 42
np.random.seed(SEED)


---
## 1. La idea

En la regresión lineal, OLS encuentra los coeficientes con una fórmula cerrada. El descenso por gradiente llega al mismo sitio dando pasos.

Se empieza con un valor cualquiera. El **gradiente** es la pendiente del costo: dice hacia qué lado el error sube. El paso va en sentido contrario, cuesta abajo.

La **tasa de aprendizaje** (`learning_rate`) no es la longitud del salto: es el factor que lo multiplica. El salto es `learning_rate` por la pendiente del costo. Lejos del mínimo la curva está empinada y el salto es grande; cerca del mínimo la pendiente es casi cero y el salto se hace chico, aunque la tasa siga valiendo lo mismo.


---
## 2. Un coeficiente, en 2D

Con un solo coeficiente el modelo es una recta que pasa por el origen:

$$\hat{y} = w\, x$$

El costo es el error cuadrático medio. Con $n$ puntos:

$$J(w) = \frac{1}{n}\sum_{i=1}^{n}(y_i - w\, x_i)^2$$

Derivando el cuadrado queda:

$$\frac{\partial J}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n}(y_i - w\, x_i)\, x_i$$

Y el paso es:

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}$$

$J(w)$ es una curva: el eje horizontal es $w$ y el vertical es el costo. Por eso esta primera gráfica es de dos dimensiones.


---
## 3. Un paso a mano

Dos puntos: $(x, y) = (1, 1)$ y $(2, 3)$. Empezamos en $w = 0$ con $\alpha = 0.1$.

Las predicciones valen 0. Los errores son $1 - 0 = 1$ y $3 - 0 = 3$.

$$\frac{\partial J}{\partial w} = -2 \cdot \frac{1\cdot 1 + 3\cdot 2}{2} = -7$$

$$w \leftarrow 0 - 0.1 \cdot (-7) = 0.7$$

El costo pasa de $\frac{1^2 + 3^2}{2} = 5$ a $\frac{(1-0.7)^2 + (3-1.4)^2}{2} = 1.325$. Un solo paso ya bajó la cuesta. El mínimo de estos dos puntos está en $w = 1.4$.


In [ ]:
x = np.array([1.0, 2.0])
y = np.array([1.0, 3.0])
w = 0.0
learning_rate = 0.1

error = y - w * x
grad_w = -2 * np.mean(error * x)
w = w - learning_rate * grad_w

print("errores:", error)
print("gradiente:", grad_w)
print("w nuevo:", w)


Los dos puntos de arriba dibujan esta curva. Cada punto marcado es un paso. El mínimo está en $w = 1.4$.

Cambia `learning_rate` y vuelve a ejecutar la celda. Con `0.2` el primer paso cae justo en el mínimo. Con `0.45` se pasa y el costo crece.


In [ ]:
learning_rate = 2
pasos_curva = 15

w = 0.0
historia_w = []
historia_costo = []
for _ in range(pasos_curva):
    error = y - w * x
    historia_w.append(w)
    historia_costo.append(np.mean(error ** 2))
    grad_w = -2 * np.mean(error * x)
    w = w - learning_rate * grad_w

pesos = np.linspace(-1, 5, 500)
costos_curva = [np.mean((y - peso * x) ** 2) for peso in pesos]

plt.figure(figsize=(7, 4))
plt.plot(pesos, costos_curva, color=palette[0])
plt.plot(historia_w, historia_costo, color=palette[1], marker="o")
plt.xlim(-1, 5)
plt.ylim(0, 5)
plt.xlabel("w")
plt.ylabel("costo")
plt.title(f"Descenso en 2D, learning_rate = {learning_rate}")
plt.show()


---
## 4. Pseudocódigo

El mismo paso, repetido. `learning_rate` es $\alpha$.

```text
learning_rate ← 0.1
w ← 0

repetir pasos veces:
    error ← y − w · x
    grad_w ← −2 · media(error · x)
    w ← w − learning_rate · grad_w
```

Con más coeficientes se añade una línea de gradiente y una línea de actualización por cada uno. El resto no cambia.


---
## 5. Una característica

Ahora el modelo tiene ordenada y pendiente. Sigue siendo **una** característica: los datos se dibujan en 2D, un eje para $x$ y otro para $y$.

$$\hat{y} = b + w\, x$$

$$\frac{\partial J}{\partial b} = -\frac{2}{n}\sum (y_i - \hat{y}_i) \qquad
\frac{\partial J}{\partial w} = -\frac{2}{n}\sum (y_i - \hat{y}_i)\, x_i$$

Con los mismos dos puntos, $b = 0$, $w = 0$ y $\alpha = 0.1$:

$$\frac{\partial J}{\partial b} = -2 \cdot \frac{1+3}{2} = -4 \qquad
b \leftarrow 0 - 0.1 \cdot (-4) = 0.4$$

$w$ se actualiza igual que antes y pasa a $0.7$.

Abajo hay 40 puntos sintéticos, $y = 1.5 + 2x$ más un poco de ruido. Cambia `learning_rate` y vuelve a ejecutar desde esta celda.


In [ ]:
rng = np.random.default_rng(SEED)
x = rng.uniform(-1, 1, 40)
y = 1.5 + 2 * x + rng.normal(0, 0.3, 40)

learning_rate = 0.2
pasos = 40

b = 0.0
w = 0.0
historia = [(b, w)]
costos = []

for _ in range(pasos):
    error = y - (b + w * x)
    costos.append(np.mean(error ** 2))
    grad_b = -2 * np.mean(error)
    grad_w = -2 * np.mean(error * x)
    b = b - learning_rate * grad_b
    w = w - learning_rate * grad_w
    historia.append((b, w))

print(f"b = {b:.3f}   w = {w:.3f}   costo final = {costos[-1]:.3f}")


In [ ]:
xs = np.linspace(x.min(), x.max(), 50)
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(x=x, y=y, color="0.45", ax=ax)
for paso in [0, 5, 15, pasos]:
    b_i, w_i = historia[paso]
    sns.lineplot(x=xs, y=b_i + w_i * xs, ax=ax, label=f"paso {paso}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Una característica: la recta en cada paso")
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(x=np.arange(1, pasos + 1), y=costos, marker="o", color=palette[0])
plt.xlabel("paso")
plt.ylabel("costo")
plt.title(f"Costo con learning_rate = {learning_rate}")
plt.show()


El costo de $b$ y $w$ ya no es una curva: es una superficie. El camino rojo es el de la celda anterior. Si cambiaste `learning_rate`, vuelve a ejecutar las dos celdas de arriba y esta.


In [ ]:
bs = np.linspace(-1, 4, 50)
ws = np.linspace(-1, 4, 50)
B, W = np.meshgrid(bs, ws)
J = np.zeros_like(B)
for i in range(B.shape[0]):
    for j in range(B.shape[1]):
        pred = B[i, j] + W[i, j] * x
        J[i, j] = np.mean((y - pred) ** 2)

camino = np.array(historia)
costo_camino = [np.mean((y - (bi + wi * x)) ** 2) for bi, wi in historia]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(B, W, J, cmap="viridis", alpha=0.75, linewidth=0)
ax.plot(camino[:, 0], camino[:, 1], costo_camino, color="red", marker="o", markersize=3)
ax.set_xlabel("b")
ax.set_ylabel("w")
ax.set_zlabel("costo")
ax.set_title("Costo de una característica, en 3D")
plt.show()


---
## 6. La tasa de aprendizaje

Misma recta, tres tasas. Una pequeña baja despacio. Una mediana llega en pocos pasos. Una grande se pasa de largo y el costo crece: los coeficientes se disparan.

> 🧪 **Prueba tú:** en la celda de la recta, prueba `learning_rate = 0.05` y después `1.2`.


In [ ]:
tasas = [0.05, 0.4, 1.2]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for ax, tasa in zip(axes, tasas):
    b_i, w_i = 0.0, 0.0
    curva = []
    for _ in range(30):
        error = y - (b_i + w_i * x)
        curva.append(np.mean(error ** 2))
        b_i = b_i - tasa * (-2 * np.mean(error))
        w_i = w_i - tasa * (-2 * np.mean(error * x))
    sns.lineplot(x=np.arange(1, 31), y=curva, marker="o", ax=ax, color=palette[0])
    ax.set_title(f"learning_rate = {tasa}")
    ax.set_xlabel("paso")
    ax.set_ylabel("costo")
plt.tight_layout()
plt.show()


---
## 7. Dos características, en 3D

Con dos características el modelo es un plano. Los datos se dibujan en 3D: $x_1$, $x_2$ y $y$.

$$\hat{y} = b + w_1 x_1 + w_2 x_2$$

Hay un gradiente más, el de $w_2$. Es el mismo cálculo de $w$, con $x_2$ en lugar de $x$:

$$\frac{\partial J}{\partial w_2} = -\frac{2}{n}\sum (y_i - \hat{y}_i)\, x_{2i}$$

```text
error ← y − (b + w1 · x1 + w2 · x2)
grad_b  ← −2 · media(error)
grad_w1 ← −2 · media(error · x1)
grad_w2 ← −2 · media(error · x2)
b  ← b  − learning_rate · grad_b
w1 ← w1 − learning_rate · grad_w1
w2 ← w2 − learning_rate · grad_w2
```

Los puntos salen de $y = 1 + 1.5\, x_1 - 2\, x_2$ más ruido.


In [ ]:
rng = np.random.default_rng(SEED)
x1 = rng.uniform(-1, 1, 40)
x2 = rng.uniform(-1, 1, 40)
y = 1 + 1.5 * x1 - 2 * x2 + rng.normal(0, 0.3, 40)

learning_rate = 0.3
pasos = 40

b = 0.0
w1 = 0.0
w2 = 0.0
historia = [(b, w1, w2)]
costos = []

for _ in range(pasos):
    error = y - (b + w1 * x1 + w2 * x2)
    costos.append(np.mean(error ** 2))
    grad_b = -2 * np.mean(error)
    grad_w1 = -2 * np.mean(error * x1)
    grad_w2 = -2 * np.mean(error * x2)
    b = b - learning_rate * grad_b
    w1 = w1 - learning_rate * grad_w1
    w2 = w2 - learning_rate * grad_w2
    historia.append((b, w1, w2))

print(f"b = {b:.3f}   w1 = {w1:.3f}   w2 = {w2:.3f}   costo final = {costos[-1]:.3f}")


In [ ]:
malla = np.linspace(-1, 1, 20)
X1, X2 = np.meshgrid(malla, malla)
Y_plano = b + w1 * X1 + w2 * X2

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(x1, x2, y, color=palette[0], depthshade=False)
ax.plot_surface(X1, X2, Y_plano, color=palette[1], alpha=0.45, linewidth=0)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("y")
ax.set_title("Dos características: los datos y el plano")
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(x=np.arange(1, pasos + 1), y=costos, marker="o", color=palette[0])
plt.xlabel("paso")
plt.ylabel("costo")
plt.title(f"Costo del plano con learning_rate = {learning_rate}")
plt.show()


---
## 8. 📋 Chuleta resumen

| Idea | Código |
|---|---|
| Predicción, una característica | `b + w * x` |
| Predicción, dos características | `b + w1 * x1 + w2 * x2` |
| Costo (error cuadrático medio) | `np.mean(error ** 2)` |
| Gradiente de la ordenada | `-2 * np.mean(error)` |
| Gradiente de un peso | `-2 * np.mean(error * x)` |
| Un paso | `w = w - learning_rate * grad_w` |
| Tasa pequeña | el costo baja, hace falta repetir más veces |
| Tasa grande | el costo crece y los coeficientes se disparan |
